### Reseta arquivos de horarios

In [ ]:
import json
import re
import os
import gdown
import fitz
import tempfile
import time
from io import BytesIO
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from upstash_vector import Index

load_dotenv()

UPSTASH_ENDPOINT = os.getenv("UPSTASH_ENDPOINT")
UPSTASH_WRITE_API_KEY = os.getenv("UPSTASH_WRITE_API_KEY")
CEREBRAS_API_KEY = os.getenv("CEREBRAS_API_KEY")

index = Index(url=UPSTASH_ENDPOINT, token=UPSTASH_WRITE_API_KEY)
cerebras_client = OpenAI(api_key=CEREBRAS_API_KEY, base_url="https://api.cerebras.ai/v1")

# URLs de download dos PDFs de horário
horario_urls_download = [
    "https://drive.google.com/uc?export=download&id=1giy2a2wZ6QMgynKqv997OjBDvp1f1WTl",
    "https://drive.google.com/uc?export=download&id=1iDkwv-NYtqDUq-fkrWeysMCqDWrcxnRV",
    "https://drive.google.com/uc?export=download&id=1kB8S27U-9gRUsC-fljykNMPxdk0nJvJm",
    "https://drive.google.com/uc?export=download&id=1u16bttceP7Hf1ampA-YeO7p3KF5hYa2h",
    "https://drive.google.com/uc?export=download&id=1RWYIwyjVJPhPotjFhz6hjymMQ2c71KUw",
    "https://drive.google.com/uc?export=download&id=16nyJTxh0MMXk8ThtqV57N-O7xeeSKqsG",
    "https://drive.google.com/uc?export=download&id=1v58tc9YycQ24oPeutaCtBW2Hah8GgNpp",
    "https://drive.google.com/uc?export=download&id=1pQIT_vhyxsq49OJ8N71owZUBTj21Hxzo",
    "https://drive.google.com/uc?export=download&id=1WppKds4GM3eH_UqquweiMlCl8o-Ty1fU",
    "https://drive.google.com/uc?export=download&id=1HuwnLNEK5lMjHZSr9sbmc62lzvziU8nh",
    "https://drive.google.com/uc?export=download&id=1zyh3AlzamjM7XppOCOPXD3WcIKHdTc9m",
]

def to_drive_view_url(url):
    if "drive.google.com/uc" in url:
        match = re.search(r"id=([a-zA-Z0-9_-]+)", url)
        if match:
            return f"https://drive.google.com/file/d/{match.group(1)}/view"
    return url

horario_urls_view = [to_drive_view_url(u) for u in horario_urls_download]

# remove do pdfs_parsed.json
with open("../data/raw/pdfs_parsed.json", "r", encoding="utf-8") as f:
    pdfs_parsed = json.load(f)

antes = len(pdfs_parsed)
pdfs_parsed = [p for p in pdfs_parsed if p["source_url"] not in set(horario_urls_download)]
print(f"pdfs_parsed: {antes} -> {len(pdfs_parsed)}")

with open("../data/raw/pdfs_parsed.json", "w", encoding="utf-8") as f:
    json.dump(pdfs_parsed, f, ensure_ascii=False, indent=2)

# remove do Upstash
urls_filter = ", ".join(f"'{u}'" for u in horario_urls_view)
index.delete(filter=f"source_url IN ({urls_filter})")

# confirma remoção
results = index.range(
    cursor="",
    limit=10,
    include_metadata=True,
    filter=f"source_url IN ({urls_filter})"
)
print(f"Pontos restantes no Upstash: {len(results.vectors)}")
print("Limpeza concluída.")

### Retorna chunks a partir de link

In [ ]:
import os
from dotenv import load_dotenv
from upstash_vector import Index

load_dotenv()

UPSTASH_ENDPOINT = os.getenv("UPSTASH_ENDPOINT")
UPSTASH_API_KEY = os.getenv("UPSTASH_API_KEY")

index = Index(url=UPSTASH_ENDPOINT, token=UPSTASH_API_KEY)

url_alvo = "https://ifrs.edu.br/canoas/ensino/professores/"

results = index.range(
    cursor="",
    limit=10,
    include_metadata=True,
    filter=f"source_url = '{url_alvo}'"
)

print(f"Chunks encontrados: {len(results.vectors)}")
for v in results.vectors:
    print(v.metadata["text"][:])
    print("---")